# Лекция 08. Итераторы и генераторы

Итератор нужен, когда данные удобнее получать по одному: читать большой отчёт построчно, забирать страницы API или обрабатывать журнал событий без списка на миллион элементов. Внутри снова нет магии — только договор о том, как запросить следующий элемент.

## Цели

После лекции вы сможете:

- различать iterable, iterator и generator;
- объяснять `iter()`, `next()` и `StopIteration`;
- писать генераторные функции с `yield` и `yield from`;
- собирать ленивые конвейеры обработки данных;
- применять генераторные выражения и основные инструменты `itertools`;
- оценивать память, однократность обхода и момент возникновения ошибок;
- распознавать неявное потребление итератора.

## Перед началом

Нужны функции, циклы, файлы и контейнеры из занятий 1–5. На протокол итерации заложено около 30 минут, на генераторы — 25 минут, на конвейеры и `itertools` — 20 минут, на контринтуитивные примеры, самопроверку и вопросы — 15 минут. Все основные примеры работают без сторонних библиотек.

## Сначала реальная задача

Экспорт операций может занимать несколько гигабайт. Если прочитать весь файл, затем построить список очищенных строк, затем список разобранных записей и только потом отфильтровать суммы, в памяти одновременно окажется несколько представлений одних данных.

Для отчёта часто достаточно маршрута «прочитать одну строку → проверить → преобразовать → при необходимости передать дальше». Итератор описывает именно такой поэлементный обмен.

## Три связанных понятия

| Понятие | Что умеет | Примеры |
|---|---|---|
| **Итерируемый объект** (`Iterable`) | отдаёт итератор по вызову `iter(obj)` | список, строка, `range`, словарь |
| **Итератор** (`Iterator`) | отдаёт следующий элемент через `next(obj)` и помнит позицию | итератор списка, открытый файл |
| **Генератор** | итератор, созданный генераторной функцией или выражением | функция с `yield`, `(x for x in data)` |

Каждый генератор является итератором, каждый итератор итерируем, но не каждый итерируемый объект является итератором.

In [ ]:
from collections.abc import Iterable, Iterator

amounts = [1200, 450, 3100]
amount_iterator = iter(amounts)

assert isinstance(amounts, Iterable)
assert not isinstance(amounts, Iterator)
assert isinstance(amount_iterator, Iterator)
assert iter(amount_iterator) is amount_iterator

## `iter()` создаёт маршрут, `next()` делает один шаг

У списка можно получить несколько независимых итераторов. Каждый хранит собственную текущую позицию. Сам список при этом не меняется.

In [ ]:
amounts = [1200, 450, 3100]
first_pass = iter(amounts)
second_pass = iter(amounts)

assert next(first_pass) == 1200
assert next(first_pass) == 450
assert next(second_pass) == 1200
assert amounts == [1200, 450, 3100]

## Завершение — это `StopIteration`

Когда элементов больше нет, `next(iterator)` поднимает `StopIteration`. Это не авария, а часть протокола. Цикл `for` ловит это исключение сам. Для одиночного запроса можно передать `next(iterator, default)` и получить значение по умолчанию вместо исключения.

In [ ]:
single = iter(["only"])
assert next(single) == "only"
assert next(single, None) is None

try:
    next(single)
except StopIteration:
    print("Итератор исчерпан")

## Что делает `for`

Цикл получает итератор один раз, затем вызывает `next()` до `StopIteration`. Поэтому `for` одинаково работает со списком, файлом, генератором и многими объектами из библиотек.

In [ ]:
source = ["income", "tax", "rent"]
iterator = iter(source)
manual_result = []

while True:
    try:
        item = next(iterator)
    except StopIteration:
        break
    manual_result.append(item.upper())

assert manual_result == [item.upper() for item in source]

## Файл уже является итератором строк

Для большого текста не нужен отдельный механизм потоковой обработки: открытый текстовый файл выдаёт строки по одной. В демонстрации `StringIO` ведёт себя как небольшой файл в памяти.

In [ ]:
from io import StringIO

report = StringIO("income;1200\ntax;300\nrent;45000\n")
assert iter(report) is report
assert next(report) == "income;1200\n"
assert next(report) == "tax;300\n"

## Генераторная функция

Если в теле функции встречается `yield`, вызов функции создаёт генератор, но не выполняет тело. Работа начинается при первом `next()` или входе в `for`. `yield value` отдаёт значение наружу и приостанавливает функцию, сохраняя локальные переменные и позицию выполнения.

In [ ]:
def iter_large_payments(payments, minimum):
    print("Начали обход")
    for payment in payments:
        if payment["amount"] >= minimum:
            yield payment

source = [
    {"id": "p-1", "amount": 300},
    {"id": "p-2", "amount": 2500},
]
stream = iter_large_payments(source, 1000)
print("Генератор создан")
assert next(stream)["id"] == "p-2"

## Состояние остаётся внутри генератора

Обычная функция при `return` заканчивается. Генератор после `yield` остаётся на паузе и продолжает со следующей строки. Поэтому накопительный итог можно выдавать после каждой операции без внешнего списка результатов.

In [ ]:
def running_balance(changes, start=0):
    balance = start
    for change in changes:
        balance += change
        yield balance

balances = running_balance([1000, -250, -100, 500], start=2000)
assert next(balances) == 3000
assert next(balances) == 2750
assert list(balances) == [2650, 3150]

## `yield` и `return` решают разные задачи

- `yield item` выдаёт один элемент и оставляет возможность продолжить.
- `return` без значения завершает генератор.
- Обычная функция может вернуть готовый список, но тогда сначала создаёт весь результат.

Генератор полезен не потому, что короче, а когда потребителю нужны элементы по одному или он может остановиться раньше конца.

## Генераторное выражение

Скобки `(expression for item in iterable if condition)` создают ленивый итератор. Синтаксис похож на list comprehension, но квадратных скобок и готового списка нет.

In [ ]:
payments = [
    {"amount": 300, "status": "pending"},
    {"amount": 2500, "status": "completed"},
    {"amount": 1700, "status": "completed"},
]
completed_amounts = (
    payment["amount"]
    for payment in payments
    if payment["status"] == "completed"
)
assert sum(completed_amounts) == 4200

## Память: контейнер и рецепт

List comprehension хранит все ссылки на результаты. Генераторное выражение хранит состояние обхода и текущий элемент. `sys.getsizeof` ниже не измеряет всю память графа объектов, но хорошо показывает разницу между готовым контейнером и компактным рецептом вычисления.

In [ ]:
from sys import getsizeof

ready = [value * 2 for value in range(100_000)]
lazy = (value * 2 for value in range(100_000))
print("Список:", getsizeof(ready))
print("Генератор:", getsizeof(lazy))
assert next(lazy) == 0

## Ленивый конвейер

Разделим работу с отчётом на маленькие этапы:

1. убрать пустые строки и комментарии;
2. разобрать значимую строку;
3. оставить подходящие записи;
4. передать результат потребителю.

Каждый этап принимает `Iterable` и выдаёт `Iterator`. В памяти одновременно находится одна строка и одна разобранная запись.

In [ ]:
from collections.abc import Iterable, Iterator

def meaningful_lines(lines: Iterable[str]) -> Iterator[str]:
    for line in lines:
        cleaned = line.strip()
        if cleaned and not cleaned.startswith("#"):
            yield cleaned

def parse_transactions(lines: Iterable[str]) -> Iterator[dict]:
    for line in lines:
        operation_id, category, raw_amount = line.split(";")
        yield {"id": operation_id, "category": category, "amount": int(raw_amount)}

def at_least(transactions: Iterable[dict], minimum: int) -> Iterator[dict]:
    for transaction in transactions:
        if transaction["amount"] >= minimum:
            yield transaction

source = StringIO("# report\nt-1;food;900\n\nt-2;rent;45000\n")
pipeline = at_least(parse_transactions(meaningful_lines(source)), 1000)
assert list(pipeline) == [{"id": "t-2", "category": "rent", "amount": 45000}]

## Раннее завершение экономит реальную работу

Если потребителю нужны первые пять ошибок, генератор может остановить чтение после пятой. Список сначала обработал бы весь источник. Лень полезна не только для памяти: она позволяет вообще не выполнять ненужную часть вычислений.

In [ ]:
def error_lines(lines):
    for line in lines:
        if "ERROR" in line:
            yield line.strip()

log = iter(["INFO start", "ERROR timeout", "INFO retry", "ERROR denied"])
errors = error_lines(log)
assert next(errors) == "ERROR timeout"
# Источник обработан только до первой найденной ошибки.

## `yield from`: передать элементы вложенного источника

API часто возвращает страницы. Если каждая страница уже итерируема, `yield from page` последовательно передаёт её элементы наружу. Для простого делегирования это точнее ручного внутреннего цикла.

In [ ]:
def iter_pages(pages):
    for page in pages:
        yield from page

pages = [[{"id": "p-1"}, {"id": "p-2"}], [], [{"id": "p-3"}]]
assert [item["id"] for item in iter_pages(pages)] == ["p-1", "p-2", "p-3"]

## `iter(callable, sentinel)`

У `iter` есть форма для функций без аргументов: вызывать функцию, пока она не вернёт специальное значение. Например, `readline` возвращает пустую строку в конце файла. Сам sentinel в результат не попадает.

In [ ]:
report = StringIO("first\nsecond\n")
lines = iter(report.readline, "")
assert [line.strip() for line in lines] == ["first", "second"]

## `itertools`: готовые детали конвейера

Модуль содержит небольшие функции, создающие итераторы. Полезный базовый набор:

- `chain(a, b, ...)` — пройти источники подряд;
- `islice(source, start, stop, step)` — ленивый срез;
- `count(start, step)` — потенциально бесконечный счётчик;
- `takewhile(predicate, source)` — брать элементы до первого нарушения;
- `pairwise(source)` — соседние пары.

Эти функции заменяют часто повторяющиеся циклы.

In [ ]:
from itertools import chain, islice

yesterday = ["event-1", "event-2"]
today = ["event-3", "event-4", "event-5"]
first_three = islice(chain(yesterday, today), 3)
assert list(first_three) == ["event-1", "event-2", "event-3"]

In [ ]:
from itertools import count, takewhile

invoice_numbers = (f"INV-{number:04d}" for number in count(101))
assert list(islice(invoice_numbers, 3)) == ["INV-0101", "INV-0102", "INV-0103"]

balances = [1000, 850, 400, -50, 300]
non_negative_prefix = takewhile(lambda balance: balance >= 0, balances)
assert list(non_negative_prefix) == [1000, 850, 400]

### Соседние события через `pairwise`

> **Появилось в Python 3.10.** До этого соседние пары часто собирали вручную через два сдвинутых итератора.

`pairwise` выдаёт пары соседних элементов. Это удобно для интервалов между событиями и проверки монотонности. Для `N` элементов получится `N - 1` пар.

In [ ]:
from itertools import pairwise

timestamps = [10, 13, 20, 21]
intervals = (right - left for left, right in pairwise(timestamps))
assert list(intervals) == [3, 7, 1]

## Потоковое слияние отсортированных источников

`heapq.merge` лениво объединяет несколько отсортированных потоков. Он не перечитывает и не сортирует все данные заново. Для словарей задаём `key`; при равных ключах сохраняется стабильный порядок источников.

In [ ]:
from heapq import merge

bank = iter([{"id": "b-1", "time": 10}, {"id": "b-2", "time": 20}])
api = iter([{"id": "a-1", "time": 15}, {"id": "a-2", "time": 30}])
merged = merge(bank, api, key=lambda event: event["time"])
assert [event["id"] for event in merged] == ["b-1", "a-1", "b-2", "a-2"]

## Аннотации описывают контракт

Функции-конвейеры обычно принимают `Iterable[T]`: им всё равно, список это, файл или другой генератор. Возвращают `Iterator[T]`, потому что результат выдаётся по одному и имеет состояние обхода. Интерфейсы берут из `collections.abc`; конкретный список в сигнатуре был бы лишним ограничением.

## Ленивость продлевает жизнь ресурса

Если генератор открывает файл внутри `with` и делает `yield`, файл остаётся открытым, пока генератор не завершится или не будет закрыт. Обычно безопаснее открывать файл снаружи и полностью потреблять конвейер внутри того же `with`.

```python
with path.open(encoding="utf-8") as source:
    for transaction in parse_transactions(meaningful_lines(source)):
        process(transaction)
```

Возвращать наружу генератор, зависящий от уже закрытого файла, нельзя.

## Когда список всё-таки лучше

Генератор не является автоматическим улучшением. Список удобнее, если нужны повторные обходы, `len()`, индексы, сортировка, случайный доступ или снимок данных на конкретный момент.

| Требование | Разумный выбор |
|---|---|
| один последовательный проход | итератор / генератор |
| ранняя остановка | генератор |
| вход больше памяти | потоковый конвейер |
| повторные обходы | материализованный контейнер |
| индексы и срезы | последовательность |
| зафиксировать текущие значения | список или кортеж |

## Неожиданно, но по правилам

Все примеры ниже следуют из двух правил: вычисление происходит при запросе следующего элемента, а итератор хранит одну изменяющуюся позицию.

### 1. Вызов генераторной функции не выполняет её тело

Даже код до первого `yield` ждёт первого `next()`. По той же причине ошибка внутри генератора может возникнуть позже и в другом месте программы — там, где результат начали потреблять.

In [ ]:
events = []

def delayed():
    events.append("started")
    yield 10

stream = delayed()
assert events == []
assert next(stream) == 10
assert events == ["started"]

### 2. Второй обход генератора может быть пустым

`iter(generator) is generator`: новый маршрут не создаётся. Первый обход сдвигает единственную позицию до конца.

In [ ]:
amounts = (value * 100 for value in [1, 2, 3])
assert list(amounts) == [100, 200, 300]
assert list(amounts) == []

### 3. Проверка `in` потребляет итератор

Поиск идёт вперёд до совпадения или конца. Найденный элемент тоже уже извлечён, поэтому последующий обход продолжится после него.

In [ ]:
operations = iter(["new", "pending", "done", "archived"])
assert "done" in operations
assert list(operations) == ["archived"]

### 4. Обычный `zip` молча останавливается на коротком входе

Это удобно для параллельного обхода, но может скрыть потерянные записи.

> **Появилось в Python 3.10.** `zip(..., strict=True)` поднимает `ValueError`, если длины различаются, и делает проверку намерения явной.

In [ ]:
clients = ["c-1", "c-2", "c-3"]
balances = [100, 200]
assert list(zip(clients, balances)) == [("c-1", 100), ("c-2", 200)]

try:
    list(zip(clients, balances, strict=True))
except ValueError:
    print("Длины источников различаются")

### 5. `any()` и `all()` могут оставить итератор наполовину пройденным

Они завершаются сразу, когда ответ уже известен. Это экономит работу, но состояние переданного итератора изменяется.

In [ ]:
checks = iter([False, False, True, False])
assert any(checks) is True
assert list(checks) == [False]

### 6. `takewhile` съедает граничный элемент

Чтобы понять, что условие впервые стало ложным, функция должна извлечь этот элемент. В результат он не попадает, но и в исходном итераторе его уже нет.

In [ ]:
source = iter([100, 80, 60, -10, 40])
prefix = takewhile(lambda value: value >= 0, source)
assert list(prefix) == [100, 80, 60]
assert list(source) == [40]

## Самопроверка

1. Чем итерируемый объект отличается от итератора?
2. Почему `iter(iterator) is iterator`?
3. Как `for` понимает, что элементы закончились?
4. В какой момент начинает выполняться генераторная функция?
5. Чем по памяти различаются list comprehension и generator expression?
6. Почему ленивый конвейер позволяет остановить чтение раньше конца файла?
7. Когда нужен `yield from`?
8. Почему список иногда лучше генератора?
9. Какие операции могут незаметно потребить часть итератора?

## Источники

- [Iterator Types — Python documentation](https://docs.python.org/3/library/stdtypes.html#iterator-types) — протокол `__iter__`, `__next__` и завершение обхода.
- [Yield expressions — Python language reference](https://docs.python.org/3/reference/expressions.html#yield-expressions) — семантика генераторных функций и `yield from`.
- [`itertools`](https://docs.python.org/3/library/itertools.html) — стандартные строительные блоки для ленивых конвейеров.
- [`collections.abc`](https://docs.python.org/3/library/collections.abc.html#collections.abc.Iterator) — интерфейсы `Iterable` и `Iterator`.

## Итоги

- `iter()` получает итератор, `next()` запрашивает элемент, `StopIteration` завершает обход.
- Генератор — итератор, который сохраняет состояние функции между `yield`.
- Ленивый конвейер обрабатывает данные по одному и допускает раннюю остановку.
- Генераторное выражение хранит рецепт вычисления, а не готовый список.
- `yield from` и `itertools` собирают сложный поток из простых частей.
- Итератор обычно одноразовый: `in`, `any`, `all`, `zip`, `islice` и другие потребители сдвигают его позицию.
- Выбор между списком и генератором определяется контрактом задачи, а не модой на «ленивость».